# 📊 Candlestick Chart — Matplotlib

Load OHLCV data from a **CSV or Parquet** file and plot a full candlestick chart with:
- Candlestick bodies + wicks (green/red)
- Volume bars
- EMA 9 / 21 / 50 overlays
- RSI(14) subplot
- Kronos pattern markers on candles
- CPR lines (pivot, BC, TC)
- Support / Resistance levels

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────

# Data source — use either parquet or csv (leave the other blank)
PARQUET_FILE = "data/NSE_RELIANCE_EQ_5m_full.parquet"  # ← change this
CSV_FILE     = ""   # e.g. "data/NSE_RELIANCE_EQ_5m_full.csv"

# Date range to plot (IST, leave blank for latest N candles)
DATE_FROM    = ""   # e.g. "2024-01-15" or "2024-01-15 09:15"
DATE_TO      = ""   # e.g. "2024-01-15 15:30"
LAST_N       = 100  # used when DATE_FROM / DATE_TO are blank

# What to show
SHOW_EMA     = True   # EMA 9, 21, 50
SHOW_RSI     = True   # RSI subplot
SHOW_CPR     = True   # Central Pivot Range lines
SHOW_SR      = True   # support / resistance
SHOW_PATTERNS = True  # pattern labels on candles

SYMBOL   = "NSE:RELIANCE-EQ"  # used in chart title only
INTERVAL = "5m"

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings("ignore")

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# dark style
plt.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#30363d",
    "axes.labelcolor":   "#8b949e",
    "xtick.color":       "#8b949e",
    "ytick.color":       "#8b949e",
    "grid.color":        "#21262d",
    "grid.linewidth":    0.5,
    "text.color":        "#c9d1d9",
    "font.size":         9,
})

C_UP   = "#3fb950"   # green
C_DOWN = "#f85149"   # red
C_EMA9  = "#58a6ff"  # blue
C_EMA21 = "#d29922"  # yellow
C_EMA50 = "#bc8cff"  # purple
C_CPR   = "#39d353"  # teal
C_SUP   = "#3fb950"  # green
C_RES   = "#f85149"  # red

print("Libraries loaded.")

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
if PARQUET_FILE:
    df = pd.read_parquet(PARQUET_FILE)
    print(f"Loaded: {PARQUET_FILE}")
elif CSV_FILE:
    df = pd.read_csv(CSV_FILE, index_col=0, parse_dates=True)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")
    print(f"Loaded: {CSV_FILE}")
else:
    raise ValueError("Set PARQUET_FILE or CSV_FILE above.")

df = df[["open","high","low","close","volume"]].astype(float)
df = df.sort_index().dropna()

# Convert to IST for display
df.index = df.index.tz_convert("Asia/Kolkata")

print(f"Total rows : {len(df):,}")
print(f"Range      : {df.index[0]}  →  {df.index[-1]}")

In [ ]:
# ── Slice to requested date range ─────────────────────────────────────────────
if DATE_FROM:
    from_ts = pd.Timestamp(DATE_FROM, tz="Asia/Kolkata")
    df = df[df.index >= from_ts]
if DATE_TO:
    to_ts = pd.Timestamp(DATE_TO, tz="Asia/Kolkata")
    df = df[df.index <= to_ts]
if not DATE_FROM and not DATE_TO:
    df = df.tail(LAST_N)

print(f"Plotting   : {len(df)} candles")
print(f"From       : {df.index[0]}")
print(f"To         : {df.index[-1]}")
df.tail(3)

In [ ]:
# ── Compute indicators ────────────────────────────────────────────────────────
from backend.agents.kronos_agent import (
    _ema, _sma, _rsi, _atr, _pivot_cpr,
    _support_resistance, _classify_candles,
)

closes  = df["close"].tolist()
candles = [
    {"open": float(r.open), "high": float(r.high),
     "low": float(r.low),  "close": float(r.close),
     "volume": float(r.volume)}
    for r in df.itertuples()
]

ema9_vals  = _ema(closes, 9)
ema21_vals = _ema(closes, 21)
ema50_vals = _ema(closes, 50)

# RSI for each bar
rsi_vals = []
for i in range(len(closes)):
    if i < 14:
        rsi_vals.append(None)
    else:
        rsi_vals.append(_rsi(closes[:i+1], 14))

# CPR from yesterday's candle (use previous day's OHLC)
cpr = {}
if SHOW_CPR and len(candles) > 1:
    cpr = _pivot_cpr(candles[:-1])

# S/R levels from the full window
sr = {"support": [], "resistance": []}
if SHOW_SR:
    sr = _support_resistance(candles, min(50, len(candles)))

# Pattern labels per candle (last 3 candles context)
pattern_labels = []
for i in range(len(candles)):
    window = candles[max(0, i-4):i+1]
    pats   = _classify_candles(window) if len(window) >= 1 else []
    # Short abbreviation
    abbr = {
        "doji": "D", "hammer": "H↑", "hanging_man": "HM",
        "inverted_hammer": "IH", "shooting_star": "SS",
        "bullish_engulfing": "BE↑", "bearish_engulfing": "BE↓",
        "morning_star": "MS↑", "evening_star": "ES↓",
        "bullish_marubozu": "MB↑", "bearish_marubozu": "MB↓",
        "three_white_soldiers": "3W", "three_black_crows": "3B",
        "bullish_harami": "BH↑", "bearish_harami": "BH↓",
        "spinning_top": "ST",
    }
    label = " ".join(abbr.get(p, p[:3]) for p in pats)
    pattern_labels.append(label)

print("Indicators computed.")
print(f"  EMA9  last : {ema9_vals[-1]:.2f}" if ema9_vals[-1] else "")
print(f"  EMA21 last : {ema21_vals[-1]:.2f}" if ema21_vals[-1] else "")
print(f"  EMA50 last : {ema50_vals[-1]:.2f}" if ema50_vals[-1] else "")
print(f"  RSI   last : {rsi_vals[-1]:.1f}"   if rsi_vals[-1] else "")
if cpr: print(f"  CPR        : pivot={cpr.get('pivot')} TC={cpr.get('tc')} BC={cpr.get('bc')}")
print(f"  Support    : {sr['support']}")
print(f"  Resistance : {sr['resistance']}")

In [ ]:
# ── Build the chart ───────────────────────────────────────────────────────────
n = len(df)
xs = np.arange(n)

# Grid: 3 rows if RSI, else 2 rows
row_heights = [3, 1, 1] if SHOW_RSI else [3, 1]
n_rows      = len(row_heights)

fig = plt.figure(figsize=(min(n * 0.12 + 4, 28), 10))
gs  = gridspec.GridSpec(n_rows, 1, figure=fig, hspace=0.06,
                         height_ratios=row_heights)

ax_candle = fig.add_subplot(gs[0])
ax_vol    = fig.add_subplot(gs[1], sharex=ax_candle)
ax_rsi    = fig.add_subplot(gs[2], sharex=ax_candle) if SHOW_RSI else None

# ── Candlesticks ──────────────────────────────────────────────────────────────
body_w = 0.6
for i, row in enumerate(df.itertuples()):
    o, h, l, c = row.open, row.high, row.low, row.close
    col = C_UP if c >= o else C_DOWN
    # Wick
    ax_candle.plot([i, i], [l, h], color=col, linewidth=0.8, zorder=2)
    # Body
    body_bot = min(o, c)
    body_h   = max(abs(c - o), 0.001 * c)   # min 0.1% height so flat candles show
    rect = mpatches.Rectangle(
        (i - body_w/2, body_bot), body_w, body_h,
        color=col, zorder=3, linewidth=0,
    )
    ax_candle.add_patch(rect)

    # Pattern label above/below candle
    if SHOW_PATTERNS and pattern_labels[i]:
        y_pos = h * 1.001 if c >= o else l * 0.999
        va    = "bottom" if c >= o else "top"
        ax_candle.text(i, y_pos, pattern_labels[i],
                       color="#d29922", fontsize=6, ha="center", va=va, zorder=5)

# ── EMA lines ─────────────────────────────────────────────────────────────────
if SHOW_EMA:
    def plot_ema(vals, color, label):
        ys = [v if v is not None else np.nan for v in vals]
        ax_candle.plot(xs, ys, color=color, linewidth=1.0, label=label, zorder=4)
    plot_ema(ema9_vals,  C_EMA9,  "EMA 9")
    plot_ema(ema21_vals, C_EMA21, "EMA 21")
    plot_ema(ema50_vals, C_EMA50, "EMA 50")

# ── CPR lines ─────────────────────────────────────────────────────────────────
if SHOW_CPR and cpr:
    for key, label, ls in [("pivot","Pivot","--"),("tc","TC",":"),("bc","BC",":")]:
        val = cpr.get(key)
        if val:
            ax_candle.axhline(val, color=C_CPR, linewidth=0.8, linestyle=ls,
                              alpha=0.7, zorder=1)
            ax_candle.text(n - 0.5, val, f" {label} {val:.2f}",
                           color=C_CPR, fontsize=7, va="center")

# ── Support / Resistance ───────────────────────────────────────────────────────
if SHOW_SR:
    for lv in sr["support"][:2]:
        ax_candle.axhline(lv, color=C_SUP, linewidth=0.7, linestyle="-.", alpha=0.5)
        ax_candle.text(0, lv, f"S {lv:.2f} ", color=C_SUP, fontsize=7, va="center", ha="right")
    for lv in sr["resistance"][:2]:
        ax_candle.axhline(lv, color=C_RES, linewidth=0.7, linestyle="-.", alpha=0.5)
        ax_candle.text(0, lv, f"R {lv:.2f} ", color=C_RES, fontsize=7, va="center", ha="right")

# ── Candle axis formatting ─────────────────────────────────────────────────────
ax_candle.set_xlim(-1, n)
ax_candle.set_title(f"{SYMBOL}  {INTERVAL}  ·  {df.index[0].strftime('%d %b %Y %H:%M')} → {df.index[-1].strftime('%d %b %Y %H:%M')} IST",
                    color="#c9d1d9", fontsize=11, pad=8)
ax_candle.grid(True, axis="y", alpha=0.4)
ax_candle.set_ylabel("Price (₹)", color="#8b949e")
plt.setp(ax_candle.get_xticklabels(), visible=False)

if SHOW_EMA:
    leg = ax_candle.legend(loc="upper left", framealpha=0.3,
                           facecolor="#161b22", labelcolor="#c9d1d9", fontsize=8)

# ── Volume ─────────────────────────────────────────────────────────────────────
vol_colors = [C_UP if df["close"].iloc[i] >= df["open"].iloc[i] else C_DOWN for i in range(n)]
ax_vol.bar(xs, df["volume"].values, color=vol_colors, alpha=0.7, width=0.8)

# Volume EMA (20)
vol_list = df["volume"].tolist()
vema20   = _ema(vol_list, 20)
vema_y   = [v if v is not None else np.nan for v in vema20]
ax_vol.plot(xs, vema_y, color="#58a6ff", linewidth=0.8, label="Vol EMA 20")

ax_vol.set_ylabel("Volume", color="#8b949e")
ax_vol.grid(True, axis="y", alpha=0.4)
ax_vol.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))
if not SHOW_RSI:
    plt.setp(ax_vol.get_xticklabels(), visible=True)
else:
    plt.setp(ax_vol.get_xticklabels(), visible=False)

# ── RSI ────────────────────────────────────────────────────────────────────────
if SHOW_RSI and ax_rsi is not None:
    rsi_y = [v if v is not None else np.nan for v in rsi_vals]
    ax_rsi.plot(xs, rsi_y, color="#bc8cff", linewidth=1.0)
    ax_rsi.axhline(70, color=C_DOWN, linewidth=0.6, linestyle="--", alpha=0.7)
    ax_rsi.axhline(30, color=C_UP,   linewidth=0.6, linestyle="--", alpha=0.7)
    ax_rsi.axhline(50, color="#30363d", linewidth=0.5)
    ax_rsi.fill_between(xs, rsi_y, 70, where=[v >= 70 if v else False for v in rsi_y],
                        color=C_DOWN, alpha=0.15)
    ax_rsi.fill_between(xs, rsi_y, 30, where=[v <= 30 if v else False for v in rsi_y],
                        color=C_UP, alpha=0.15)
    ax_rsi.set_ylim(0, 100)
    ax_rsi.set_ylabel("RSI", color="#8b949e")
    ax_rsi.grid(True, axis="y", alpha=0.4)
    ax_rsi.text(n - 1, 72, "OB", color=C_DOWN, fontsize=7, va="bottom", ha="right")
    ax_rsi.text(n - 1, 28, "OS", color=C_UP,   fontsize=7, va="top",    ha="right")
    # Current RSI value label
    cur_rsi = next((v for v in reversed(rsi_vals) if v is not None), None)
    if cur_rsi:
        ax_rsi.text(n - 0.5, cur_rsi, f" {cur_rsi:.1f}",
                    color="#bc8cff", fontsize=8, va="center")

# ── X-axis timestamps ─────────────────────────────────────────────────────────
ax_bottom = ax_rsi if SHOW_RSI and ax_rsi else ax_vol

# Show ticks every ~20 candles (or every day boundary)
step = max(1, n // 15)
tick_idx    = list(range(0, n, step))
tick_labels = [df.index[i].strftime("%d %b\n%H:%M") for i in tick_idx]
ax_bottom.set_xticks(tick_idx)
ax_bottom.set_xticklabels(tick_labels, fontsize=7, color="#8b949e")

plt.tight_layout()
plt.subplots_adjust(hspace=0.05)

# Save
safe = SYMBOL.replace(":","_").replace("-","_")
out_png = Path("data") / f"chart_{safe}_{INTERVAL}.png"
Path("data").mkdir(exist_ok=True)
plt.savefig(out_png, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Chart saved → {out_png}")

In [ ]:
# ── Plot a specific single trading day ───────────────────────────────────────
# Change this date and run to see any day in your data

PLOT_DATE = "2024-06-10"   # YYYY-MM-DD in IST

day_mask = df.index.normalize() == pd.Timestamp(PLOT_DATE, tz="Asia/Kolkata").normalize()
day_df   = df[day_mask]

if day_df.empty:
    print(f"No data for {PLOT_DATE} — try a weekday when market was open.")
else:
    print(f"{PLOT_DATE}  —  {len(day_df)} candles  "
          f"({day_df.index[0].strftime('%H:%M')} → {day_df.index[-1].strftime('%H:%M')} IST)")

    day_closes  = day_df["close"].tolist()
    day_candles = [{"open":float(r.open),"high":float(r.high),
                    "low":float(r.low),"close":float(r.close),
                    "volume":float(r.volume)} for r in day_df.itertuples()]

    day_e9  = _ema(day_closes, 9)
    day_e21 = _ema(day_closes, 21)

    fig2, (ax2c, ax2v) = plt.subplots(2, 1, figsize=(18, 7), sharex=True,
                                       gridspec_kw={"height_ratios":[3,1]})
    xs2 = np.arange(len(day_df))

    for i, row in enumerate(day_df.itertuples()):
        o, h, l, c = row.open, row.high, row.low, row.close
        col = C_UP if c >= o else C_DOWN
        ax2c.plot([i,i],[l,h], color=col, linewidth=0.9)
        bh = max(abs(c - o), 0.001 * c)
        ax2c.add_patch(mpatches.Rectangle((i-0.35, min(o,c)), 0.7, bh,
                                           color=col, linewidth=0))

    # EMA 9 + 21
    ax2c.plot(xs2, [v or np.nan for v in day_e9],  color=C_EMA9,  linewidth=1, label="EMA 9")
    ax2c.plot(xs2, [v or np.nan for v in day_e21], color=C_EMA21, linewidth=1, label="EMA 21")

    # VWAP
    tp   = (day_df["high"] + day_df["low"] + day_df["close"]) / 3
    vwap = (tp * day_df["volume"]).cumsum() / day_df["volume"].cumsum()
    ax2c.plot(xs2, vwap.values, color="#f0883e", linewidth=1, linestyle="--", label="VWAP")

    ax2c.set_title(f"{SYMBOL}  {INTERVAL}  —  {PLOT_DATE}", color="#c9d1d9", fontsize=11)
    ax2c.grid(True, axis="y", alpha=0.4)
    ax2c.legend(loc="upper left", framealpha=0.3, facecolor="#161b22",
                labelcolor="#c9d1d9", fontsize=8)
    ax2c.set_ylabel("Price (₹)", color="#8b949e")

    vc = [C_UP if day_df["close"].iloc[i] >= day_df["open"].iloc[i] else C_DOWN for i in range(len(day_df))]
    ax2v.bar(xs2, day_df["volume"].values, color=vc, alpha=0.7, width=0.8)
    ax2v.set_ylabel("Volume", color="#8b949e")
    ax2v.grid(True, axis="y", alpha=0.4)
    ax2v.yaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))

    # Time labels every 30 min
    time_ticks = [i for i, ts in enumerate(day_df.index) if ts.minute in (15, 45) or
                  (ts.minute == 0 and ts.hour in range(9,16))]
    ax2v.set_xticks(time_ticks)
    ax2v.set_xticklabels([day_df.index[i].strftime("%H:%M") for i in time_ticks],
                         fontsize=8, color="#8b949e")

    plt.tight_layout()
    out2 = Path("data") / f"chart_{safe}_{INTERVAL}_{PLOT_DATE}.png"
    plt.savefig(out2, dpi=150, bbox_inches="tight", facecolor=fig2.get_facecolor())
    plt.show()
    print(f"Chart saved → {out2}")